# HCP Engagement Insight Analyzer

## Objetivos de Aprendizaje

In this notebook you'll learn how to:
1. Build an **HCP engagement warehouse** with Snowflake SQL
2. Score call notes with `SNOWFLAKE.CORTEX.SENTIMENT()`
3. Extract structured intents using `AI_EXTRACT` (Snowflake Cortex)
4. Summarize themes + next best actions for district leads
5. Deliver a guardrailed Streamlit dashboard for medical/commercial leadership

---

## What You'll Build
- Cortex-enriched HCP interaction tables (sentiment + extracted intents)
- SQL analytics for priority scoring and topic frequency
- Streamlit dashboard with dependency checks and CSV exports

**Estimated runtime:** ~12 minutes on `SMALL` warehouse

---

In [ ]:
-- Environment setup
CREATE DATABASE IF NOT EXISTS HCP_ENGAGEMENT_DB;
CREATE SCHEMA IF NOT EXISTS HCP_ENGAGEMENT_DB.PUBLIC;
USE SCHEMA HCP_ENGAGEMENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
  WITH WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS database_name,
       CURRENT_SCHEMA() AS schema_name,
       CURRENT_WAREHOUSE() AS warehouse_name;

---

## Paso 1: Create Core Tables

In [ ]:
-- HCP master data
CREATE OR REPLACE TABLE hcp_profiles (
    hcp_id VARCHAR(50) PRIMARY KEY,
    hcp_name VARCHAR(120),
    specialty VARCHAR(80),
    region VARCHAR(50),
    segment VARCHAR(40),
    preferred_channel VARCHAR(40)
);

-- Interaction summary
CREATE OR REPLACE TABLE engagement_interactions (
    interaction_id VARCHAR(50) PRIMARY KEY,
    hcp_id VARCHAR(50),
    exec_owner VARCHAR(120),
    interaction_type VARCHAR(40),
    channel VARCHAR(40),
    audience VARCHAR(40),
    topic VARCHAR(80),
    interaction_timestamp TIMESTAMP,
    follow_up_status VARCHAR(40)
);

-- Call notes
CREATE OR REPLACE TABLE hcp_call_notes (
    note_id VARCHAR(50) PRIMARY KEY,
    interaction_id VARCHAR(50),
    hcp_id VARCHAR(50),
    call_summary STRING,
    commitments STRING
);

-- Cortex sentiment table
CREATE OR REPLACE TABLE engagement_sentiment (
    sentiment_id VARCHAR(50) PRIMARY KEY,
    note_id VARCHAR(50),
    interaction_id VARCHAR(50),
    hcp_id VARCHAR(50),
    sentiment_score FLOAT,
    sentiment_label VARCHAR(40),
    sentiment_magnitude FLOAT,
    analyzed_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Cortex extracted intents
CREATE OR REPLACE TABLE engagement_insights (
    insight_id VARCHAR(50) PRIMARY KEY,
    note_id VARCHAR(50),
    hcp_id VARCHAR(50),
    primary_intent STRING,
    requested_support STRING,
    urgency_signal STRING,
    next_best_action STRING,
    generated_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Aggregated metrics
CREATE OR REPLACE TABLE engagement_metrics (
    metric_id VARCHAR(50) PRIMARY KEY,
    metric_date DATE,
    metric_name VARCHAR(60),
    metric_value FLOAT,
    trend_label VARCHAR(20),
    generated_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

---

## Paso 2: Generar Datos Sintéticos HCP Data

In [ ]:
-- HCP profiles
INSERT INTO hcp_profiles
SELECT 
    'HCP_' || LPAD(SEQ4()::VARCHAR, 4, '0') AS hcp_id,
    CONCAT(first_name, ' ', last_name) AS hcp_name,
    specialty,
    region,
    segment,
    preferred_channel
FROM (
    SELECT value:first_name::string AS first_name,
           value:last_name::string AS last_name
    FROM TABLE(FLATTEN(INPUT => PARSE_JSON('[
        {"first_name":"Emma","last_name":"Larsen"},
        {"first_name":"Noah","last_name":"Patel"},
        {"first_name":"Sofie","last_name":"Madsen"},
        {"first_name":"Lucas","last_name":"Hansen"},
        {"first_name":"Olivia","last_name":"Nguyen"},
        {"first_name":"Maya","last_name":"Singh"},
        {"first_name":"Victor","last_name":"Andresen"},
        {"first_name":"Liam","last_name":"Eriksen"},
        {"first_name":"Ella","last_name":"Johansson"},
        {"first_name":"Mikkel","last_name":"Krogh"}
    ]')))
) names,
LATERAL (SELECT * FROM (VALUES ('Endocrinology'),('Primary Care'),('Metabolic Specialist'),('Diabetes Educator'))) sp(specialty),
LATERAL (SELECT * FROM (VALUES ('Nordics'),('US East'),('US West'),('Germany'),('France'))) rg(region),
LATERAL (SELECT * FROM (VALUES ('Tier 1'),('Tier 2'),('Strategic'))) seg(segment),
LATERAL (SELECT * FROM (VALUES ('In-person'),('Virtual'),('Email'))) ch(preferred_channel)
LIMIT 60;

-- Interactions + call notes
INSERT INTO engagement_interactions
WITH seq AS (SELECT SEQ4() AS idx FROM TABLE(GENERATOR(ROWCOUNT => 200)))
SELECT 
    'INT_' || LPAD(idx::VARCHAR, 5, '0') AS interaction_id,
    h.hcp_id,
    'Field Director ' || LPAD(UNIFORM(1, 20, RANDOM())::VARCHAR, 2, '0') AS exec_owner,
    CASE MOD(idx,5)
        WHEN 0 THEN 'Virtual Detail'
        WHEN 1 THEN 'Office Visit'
        WHEN 2 THEN 'Speaker Program'
        WHEN 3 THEN 'Medical Inquiry'
        ELSE 'Lunch & Learn'
    END AS interaction_type,
    CASE MOD(idx,4)
        WHEN 0 THEN 'Teams'
        WHEN 1 THEN 'On-site'
        WHEN 2 THEN 'Email'
        ELSE 'Webinar'
    END AS channel,
    CASE MOD(idx,3)
        WHEN 0 THEN 'Specialist'
        WHEN 1 THEN 'Primary Care'
        ELSE 'Clinic'
    END AS audience,
    CASE MOD(idx,5)
        WHEN 0 THEN 'Obesity'
        WHEN 1 THEN 'Cardiometabolic'
        WHEN 2 THEN 'Access & Reimbursement'
        WHEN 3 THEN 'Clinical Evidence'
        ELSE 'Supply Chain'
    END AS topic,
    DATEADD('day', -UNIFORM(1, 45, RANDOM()), CURRENT_TIMESTAMP()) AS interaction_timestamp,
    CASE MOD(idx,3)
        WHEN 0 THEN 'open'
        WHEN 1 THEN 'scheduled'
        ELSE 'closed'
    END AS follow_up_status
FROM seq
CROSS JOIN hcp_profiles h
QUALIFY ROW_NUMBER() OVER (ORDER BY idx) <= 200;

INSERT INTO hcp_call_notes
SELECT 
    'NOTE_' || LPAD(ROW_NUMBER() OVER (ORDER BY i.interaction_timestamp)::VARCHAR, 6, '0') AS note_id,
    i.interaction_id,
    i.hcp_id,
    CASE MOD(ROW_NUMBER() OVER (ORDER BY i.interaction_timestamp),4)
        WHEN 0 THEN 'HCP asked about expanded Xarelto supply and payer coverage hurdles.'
        WHEN 1 THEN 'Discussion focused on cardiometabolic outcomes; requested additional CVOT summaries.'
        WHEN 2 THEN 'Physician highlighted patient delays due to prior authorizations; asked for updated reimbursement tools.'
        ELSE 'Clinic interested in obesity patient journey toolkit; wants co-pay data and starter kits.'
    END AS call_summary,
    CASE MOD(ROW_NUMBER() OVER (ORDER BY i.interaction_timestamp),3)
        WHEN 0 THEN 'Share payer toolkit and follow up next week.'
        WHEN 1 THEN 'Email new CVOT deck and schedule speaker event.'
        ELSE 'Escalate access issue to reimbursement team.'
    END AS commitments
FROM engagement_interactions i;

---

## Paso 3: Apply `SNOWFLAKE.CORTEX.SENTIMENT()`

In [ ]:
CREATE OR REPLACE TABLE engagement_sentiment AS
WITH ranked_notes AS (
    SELECT 
        c.note_id,
        c.interaction_id,
        c.hcp_id,
        c.call_summary,
        ROW_NUMBER() OVER (ORDER BY i.interaction_timestamp DESC, c.note_id) AS rn
    FROM hcp_call_notes c
    JOIN engagement_interactions i ON c.interaction_id = i.interaction_id
)
SELECT 
    'SENT_' || LPAD(rn::VARCHAR, 6, '0') AS sentiment_id,
    note_id,
    interaction_id,
    hcp_id,
    SNOWFLAKE.CORTEX.SENTIMENT(call_summary) AS sentiment_score,
    CASE 
        WHEN sentiment_score >= 0.6 THEN 'Highly Positive'
        WHEN sentiment_score >= 0.2 THEN 'Positive'
        WHEN sentiment_score > -0.2 THEN 'Neutral'
        WHEN sentiment_score > -0.6 THEN 'Negative'
        ELSE 'Critical'
    END AS sentiment_label,
    ROUND(ABS(sentiment_score),3) AS sentiment_magnitude,
    CURRENT_TIMESTAMP() AS analyzed_timestamp
FROM ranked_notes
QUALIFY rn <= 150;

SELECT sentiment_label, COUNT(*) AS notes
FROM engagement_sentiment
GROUP BY sentiment_label
ORDER BY notes DESC;

---

## Paso 4: Extract Intent with `AI_EXTRACT` (Cortex)

In [ ]:
CREATE OR REPLACE TABLE engagement_insights AS
WITH risky_notes AS (
    SELECT 
        n.note_id,
        n.hcp_id,
        n.call_summary,
        s.sentiment_label,
        ROW_NUMBER() OVER (ORDER BY s.sentiment_score ASC) AS rn
    FROM hcp_call_notes n
    JOIN engagement_sentiment s ON n.note_id = s.note_id
),
extracted AS (
    SELECT 
        'INSIGHT_' || LPAD(rn::VARCHAR, 6, '0') AS insight_id,
        note_id,
        hcp_id,
        AI_EXTRACT(
            text => call_summary, 
            responseFormat => {
                'primary_intent': 'What is the primary intent of the healthcare provider?',
                'requested_support': 'What support or resources are being requested?',
                'urgency_signal': 'How urgent is this request? Respond with High, Medium, or Low.',
                'next_best_action': 'Recommend the next best action for the Bayer field team in under 20 words.'
            }
        ) AS extracted_json,
        CURRENT_TIMESTAMP() AS generated_timestamp
    FROM risky_notes
    QUALIFY rn <= 100
)
SELECT 
    insight_id,
    note_id,
    hcp_id,
    extracted_json:primary_intent::STRING AS primary_intent,
    extracted_json:requested_support::STRING AS requested_support,
    extracted_json:urgency_signal::STRING AS urgency_signal,
    extracted_json:next_best_action::STRING AS next_best_action,
    generated_timestamp
FROM extracted;

SELECT urgency_signal, COUNT(*) AS insights
FROM engagement_insights
GROUP BY urgency_signal;

---

## Paso 5: Analytical Views & Metrics

In [ ]:
CREATE OR REPLACE VIEW vw_hcp_engagement_priority AS
SELECT 
    i.hcp_id,
    p.hcp_name,
    p.specialty,
    p.region,
    i.interaction_type,
    i.topic,
    s.sentiment_label,
    s.sentiment_score,
    ins.primary_intent,
    ins.requested_support,
    ins.urgency_signal,
    ins.next_best_action,
    CASE 
        WHEN s.sentiment_label IN ('Negative','Critical') OR ins.urgency_signal = 'High' THEN '⚠️ Follow-up within 48h'
        WHEN s.sentiment_label = 'Neutral' THEN '⏳ Monitor'
        ELSE '✅ Healthy'
    END AS recommended_path
FROM engagement_interactions i
JOIN engagement_sentiment s ON i.interaction_id = s.interaction_id
JOIN hcp_profiles p ON i.hcp_id = p.hcp_id
LEFT JOIN engagement_insights ins ON s.note_id = ins.note_id;

CREATE OR REPLACE TABLE engagement_metrics AS
SELECT 
    'METRIC_' || LPAD(ROW_NUMBER() OVER (ORDER BY DATE(analyzed_timestamp))::VARCHAR, 6, '0') AS metric_id,
    DATE(analyzed_timestamp) AS metric_date,
    'daily_avg_sentiment' AS metric_name,
    ROUND(AVG(sentiment_score),3) AS metric_value,
    CASE 
        WHEN AVG(sentiment_score) > LAG(AVG(sentiment_score)) OVER (ORDER BY DATE(analyzed_timestamp)) THEN 'up'
        WHEN AVG(sentiment_score) < LAG(AVG(sentiment_score)) OVER (ORDER BY DATE(analyzed_timestamp)) THEN 'down'
        ELSE 'flat'
    END AS trend_label,
    CURRENT_TIMESTAMP() AS generated_timestamp
FROM engagement_sentiment
GROUP BY DATE(analyzed_timestamp);

---

## Paso 6: Summary Queries

In [ ]:
SELECT hcp_name, region, specialty, urgency_signal, COUNT(*) AS requests
FROM vw_hcp_engagement_priority
WHERE urgency_signal = 'High'
GROUP BY 1,2,3,4
ORDER BY requests DESC;

SELECT primary_intent, COUNT(*) AS notes
FROM engagement_insights
GROUP BY primary_intent
ORDER BY notes DESC
LIMIT 10;

SELECT metric_date, metric_value, trend_label
FROM engagement_metrics
ORDER BY metric_date DESC;

---

## Paso 7: Dashboard Streamlit

In [ ]:
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title("🩺 HCP Engagement Insight Analyzer")
st.caption("Cortex AI-powered prioritization for Bayer field teams")

required_tables = [
    'HCP_PROFILES',
    'ENGAGEMENT_INTERACTIONS',
    'ENGAGEMENT_SENTIMENT',
    'ENGAGEMENT_INSIGHTS',
    'ENGAGEMENT_METRICS',
    'VW_HCP_ENGAGEMENT_PRIORITY'
]
missing = []
for table in required_tables:
    try:
        session.sql(f"SELECT 1 FROM {table} LIMIT 1").collect()
    except Exception:
        missing.append(table)

if missing:
    st.error("Missing objects: " + ", ".join(missing))
    st.info("Run all SQL cells above before launching the dashboard.")
    st.stop()

summary = session.sql("""
    SELECT 
        (SELECT COUNT(*) FROM ENGAGEMENT_INTERACTIONS) AS interactions,
        (SELECT COUNT(DISTINCT hcp_id) FROM HCP_PROFILES) AS hcps,
        (SELECT ROUND(AVG(sentiment_score),3) FROM ENGAGEMENT_SENTIMENT) AS avg_sentiment,
        (SELECT COUNT(*) FROM ENGAGEMENT_INSIGHTS) AS llm_insights
""").to_pandas().iloc[0]

col1, col2, col3, col4 = st.columns(4)
col1.metric("Interactions", f"{int(summary['INTERACTIONS']):,}")
col2.metric("HCPs", int(summary['HCPS']))
col3.metric("Avg Sentiment", summary['AVG_SENTIMENT'])
col4.metric("LLM Insights", int(summary['LLM_INSIGHTS']))

st.markdown("---")
tab1, tab2, tab3 = st.tabs(["📈 Trends", "🚨 Urgent Accounts", "📋 Call Notes"])

with tab1:
    st.subheader("Daily Sentiment Trend")
    metrics_df = session.sql("""
        SELECT TO_CHAR(metric_date, 'YYYY-MM-DD') AS metric_date, metric_value, trend_label
        FROM ENGAGEMENT_METRICS
        ORDER BY metric_date
    """).to_pandas()
    if metrics_df.empty:
        st.info("No metrics yet. Run the metric cell above.")
    else:
        metrics_df['METRIC_DATE'] = pd.to_datetime(metrics_df['METRIC_DATE'], errors='coerce')
        chart = alt.Chart(metrics_df).mark_line(point=True).encode(
            x=alt.X('METRIC_DATE:T', title='Date'),
            y=alt.Y('METRIC_VALUE:Q', title='Avg Sentiment'),
            color=alt.value('#1B6CA8'),
            tooltip=[alt.Tooltip('METRIC_DATE:T', title='Date'), 'METRIC_VALUE', 'TREND_LABEL']
        ).properties(height=340)
        st.altair_chart(chart, use_container_width=True)
        st.dataframe(metrics_df, use_container_width=True)

with tab2:
    st.subheader("Urgent Follow-ups (<48h)")
    urgent_df = session.sql("""
        SELECT hcp_name, region, specialty, topic, sentiment_label, urgency_signal, next_best_action
        FROM VW_HCP_ENGAGEMENT_PRIORITY
        WHERE urgency_signal = 'High'
        ORDER BY sentiment_label, hcp_name
        LIMIT 150
    """).to_pandas()
    if urgent_df.empty:
        st.success("No urgent accounts right now!")
    else:
        st.dataframe(urgent_df, use_container_width=True)
        csv = urgent_df.to_csv(index=False)
        st.download_button("Download Urgent List", csv, "hcp_urgent_accounts.csv")

with tab3:
    st.subheader("Detailed Call Notes")
    limit = st.slider("Records", 10, 200, 50)
    notes_df = session.sql(f"""
        SELECT 
            i.interaction_timestamp,
            p.hcp_name,
            i.topic,
            s.sentiment_label,
            ins.primary_intent,
            ins.requested_support,
            ins.next_best_action,
            c.call_summary
        FROM HCP_CALL_NOTES c
        JOIN ENGAGEMENT_INTERACTIONS i ON c.interaction_id = i.interaction_id
        JOIN HCP_PROFILES p ON c.hcp_id = p.hcp_id
        JOIN ENGAGEMENT_SENTIMENT s ON c.note_id = s.note_id
        LEFT JOIN ENGAGEMENT_INSIGHTS ins ON c.note_id = ins.note_id
        ORDER BY i.interaction_timestamp DESC
        LIMIT {limit}
    """).to_pandas()
    st.dataframe(notes_df, use_container_width=True)
    csv = notes_df.to_csv(index=False)
    st.download_button("Download Call Notes", csv, "hcp_call_notes.csv")

st.markdown("---")
st.success("✅ Dashboard operational | Data current")

---


In [ ]:
-- Uncomment to reset environment

-- DROP DATABASE IF EXISTS HCP_ENGAGEMENT_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;